---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Homework 6**: Multimodal Image Search

### 📅 **Due Date**: Day of Lecture 9, 11:59 PM

### Difficulty: ★★★★☆

**Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

### Instructions

1. Use any external resource or collaborate with your classmates. But you have to cite external resources you've used and mention classmates you've collaborated with.
2. Run all cells before submitting to ensure they work.
3. Some tasks involve LLM API calls that cost money. Budget wisely.
4. I suggest you save intermediate results (descriptions, metadata) to a `temp/` folder so that you don't re-run expensive API calls.

## 1. Setup & Explore

The `../data/dogs/` directory contains dog images.
- These include different breeds, artistic styles (photographs, sketches, paintings, cartoons), settings, and some images with text or unusual content.

### Load and display images

Load all `.png` file paths from `../data/dogs/`, sorted by name. 
- Explore the data
- How many images are there? Are they AI generated? What type of variety do you observe? What breeds do you see? What artistic styles? Any images that look unusual or have text?

In [ ]:
import os
import base64

import litellm
from litellm import completion

from pydantic import BaseModel, Field
from typing import Literal, Callable

import asyncio

import time
import tqdm

from sentence_transformers import SentenceTransformer
import numpy as np

from collections import Counter
from typing import Callable

In [ ]:
# Taking length of directory listing
dir = "C:/Users/leona/ce/data/dogs"     # dir for directory

pics_qty = len(os.listdir(dir))     # qty for quantity
print(f"There are {pics_qty} images supplied")

* _Per recollection of instructor disclosure, all of the images were generated using Gemini._ 

* _I observe variety in_
    * _style (photorealistic, monochromatic drawing, polychromatic painting, cartoon);_
    * _setting (home interior, urban built environment, wilderness);_
    * _composition (frontal and three-quarters views; mutliple canine and/or human figures; framing devices; accessories like coffee, toys)._

* _Drawing on extant domain knowledge– on aspects like coat pattern and craniofacial anatomy– and having consulted listings from the authoritative source of [American Kennel Club](https://www.akc.org/), I believe the images depict ten breeds, as follows_

| _**Filename number range<br>(inclusive)**_ | _**Breed**_ |
| - | - |
| _001 through 010_ | _[Golden Retriever](https://www.akc.org/dog-breeds/golden-retriever/)_ |
| _011 through 020_ | _[Pug](https://www.akc.org/dog-breeds/pug/)_ |
| _021 through 030_ | _[German Shepherd](https://www.akc.org/dog-breeds/german-shepherd-dog/)_ |
| _031 through 040_ | _[Husky](https://www.akc.org/dog-breeds/siberian-husky/)_ |
| _041 through 050_ | _[Corgi](https://www.akc.org/dog-breeds/cardigan-welsh-corgi/)_ |
| _051 through 060_ | _[Dalmatian](https://www.akc.org/dog-breeds/dalmatian/)_ |
| _061 through 070_ | _[Bulldog](https://www.akc.org/dog-breeds/bulldog/) <br> (I initially misidentified as Boxer)_ |
| _071 through 080_ | _[Shiba Inu](https://www.akc.org/dog-breeds/shiba-inu/)_ |
| _081 through 090_ | _[Beagle](https://www.akc.org/dog-breeds/beagle/)_ |
| _091 through 100_ | _[Poodle](https://www.akc.org/dog-breeds/poodle-standard/)_ |

* _Setting aside my knowledge of their synthetic provenance, none of these images strike me at first glance as visually "unusual." Closer inspection does reveal some irregularities: calendars are incorrect, misspelled or otherwise garbled words._

* _Several images feature text, primarily with reference to the year 2026._

## 2. The Task

Your goal is to build a **searchable index** over these dog images that allows us to find relevant images using natural language queries, metadata filters, or both.

There are broadly two approaches:

**Approach A: Text-based pipeline**
1. Use a vision LLM to generate text descriptions of each image
2. Extract structured metadata (breed, style, setting, etc.) using an LLM with structured output
3. Embed the descriptions and build a searchable index (e.g., using LanceDB)
4. Search using semantic similarity over text + metadata filters

**Approach B: Multimodal embeddings**
1. Use a multimodal embedding model to embed the images directly
2. Embed text queries in the same space
3. Search by comparing query embeddings to image embeddings

You may combine both approaches or use a different method entirely. Use any tools from the course — `litellm`, `lancedb`, `pydantic`, `sentence-transformers` - or any other tools you want.


### A1. Engaging vision LLM

_**Decision**<br>_
_I'll undertake Approach A here, accessing OpenAI models through LiteLLM._

In [ ]:
os.environ.get("OPENAI_API_KEY")
print(f"Length of OpenAI API key: {len(os.environ["OPENAI_API_KEY"])} characters")

_**Background research on possible technical constraints**<br>_
_I read that_
* _some OpenAI models use [patch-based image tokenzation](https://developers.openai.com/api/docs/guides/images-vision#patch-based-image-tokenization), with each patch comprising 32 $\times$ 32 pixels. This works nicely on the images in question, which are 1024 $\times$ 1024 pixels each (making for 1024 patches, below_  `gpt-5-nano`_'s ceiling of 1536); and_
* _there is a 512MB [batch payload restriction](https://developers.openai.com/api/docs/guides/images-vision#image-input-requirements)._

In [ ]:
# Assigning folder path, initializing size
size = 0

# Iterating to tally total size
for picture in os.scandir(dir):
    size += os.path.getsize(picture)
    
if size <= (512 * (1024 ** 2)):     # Converting from bytes to MB
    print(f"At {size / (1024 ** 2):,.1f} MB, folder fits in single payload")

_**Test on first image**<br>_
_I take the first image as a partial "proof of concept," reserving output structuring for later consideration._

_I didn't find the LiteLLM documentation to be very clear on how to handle images that aren't situated at publicly accessible URLs, so I went to Gemini to ask about uploading local files. The following cell is lightly modified from what Gemini generated._

In [ ]:
# Defining helper function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

# Establishing filepath for file upload
    # Using listdir(ectory) index in anticipation of iterative operation
image_path = ("C:/Users/leona/ce/data/dogs/" 
    + os.listdir("C:/Users/leona/ce/data/dogs")[0])

base64_image = encode_image(image_path)

response_schemaless = litellm.completion(
    model = "gpt-5-nano",
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Describe this local file."
                    },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{base64_image}"
                        }     # closing image_url
                    }      # closing image
                ]     # closing content
            }     # closing messsages[0]
        ]     # closing messages
    )     # closing completion

In [ ]:
# Printing to validate output
print(response_schemaless.model_dump_json(indent = 2))

# Retaining stochastic output by writing to external file
with open("test_response.txt", "a", encoding = "utf-8") as tr:
    tr.write("\n\n" + 
    str(response_schemaless.choices[0].message.content))

_Pretty impressive analysis, I'd say._

### A2. Structuring metadata
_Now: I define a Pydantic schema to better support the search._

In [ ]:
class DogPic(BaseModel):
    """Structured representation of elements in a dog picture."""
    
    breed: str = Field(
        description = "Dog breed pictured")
    number: int = Field(
        description = "Number of dogs in image",
        ge = 1)    # Setting threshold of at least one whole dog
    age: Literal["puppy", "adult"] = Field(
        description = "Coarse age bucketing")
    setting: str = Field(
        description = "Environment surrounding dog")
    accessories: list[str] = Field(
        description = "Canine-related lifestyle items present, or being worn or played with",
        min_length = 0, max_length = 3)     # Possibly none but no more than three characters
    mood: str = Field(
        description = "Emotional tone, based on dog's expression and posture")
    visual_style: str = Field(
        description = "Aesthetic and/or artistic look/style/(simulated) medium")
    text: str | None = Field(
        default = None,     # Providing default argument for optional schematic element
        description = "Textual content (graphemes) appearing anywhere in the image (OPTIONAL)")
    botanicals: str | None = Field(
        default = None,     # Providing default argument for optional schematic element
        description = "Flowers or trees depicted (OPTIONAL)")

    pass

In [ ]:
async def async_extract_dogpic(image_base64: base64) -> DogPic:
    """Extract elements of a single dog picture asynchronously."""

    response = await litellm.acompletion(
        model = "openai/gpt-5-nano",
        messages = [
            {
                "role" : "system",
                "content" : "Image processor decomposing image containing a dog into assorted visual elements and returning said elements per Pydantic schema"
                },
                
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Please describe this dog picture, present(ed) as local file."
                        },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{image_base64}"
                            }     # closing image_url
                        }     # closing image
                    ]     # closing content
                }     # closing user
            ],     # closing messages
            response_format = DogPic
        )     # closing acompletion

    dogPic_data = DogPic.model_validate_json(response.choices[0].message.content)

    return dogPic_data

In [ ]:
# Arranging extraction across full dataset
async def extract_all_dogpics(dir: str) -> list[DogPic]:
    """Process all dog pictures concurrently and return results."""
    
    tasks = [async_extract_dogpic(encode_image(dir + "/" + filename)) for filename in os.listdir(dir)]

    return await asyncio.gather(*tasks)

In [ ]:
# Processing images
start = time.time()
pics_parsed = []
pics_parsed = await extract_all_dogpics("C:/Users/leona/ce/data/dogs")
elapsed = time.time() - start

# Retaining stochastic output by writing to another external file
with open("responses-with-schema.txt", "a", encoding = "utf-8") as rws:
    for parsing in pics_parsed:
        rws.write(str(parsing))

# Issuing task completion update
print(f"Processed {pics_qty} dog pictures in {elapsed:.2f} seconds")

### A3. Embedding descriptions, searchable index

In [ ]:
# Embedding through local model
local_model = "all-MiniLM-L6-v2"
model = SentenceTransformer(local_model)

pics_embeds = []
for parsing in pics_parsed:
    pics_embeds.append(model.encode(str(parsing)))

pics_embeds = np.array(pics_embeds)

In [ ]:
print(pics_embeds.shape)
np.save("temp/hw6_embeddings.npy", pics_embeds)

In [16]:
from helpers import (
    snowball_tokenize, build_index,
    )

### A4. Search by semantic similarity and metadata

In [17]:
from helpers import (
    batch_cosine_similarity, semantic_search,
    evaluate_search
    )
    

## 3. Your Own Queries

To make a good search system, you will need to come up with some of your own queries and "golden" answers to these queries to evaluate your system with. Show us your work.

In [ ]:
# Checking 10 images throughout set with particular qualities
display(pics_parsed[5],  # checking for greyscale drawing
    pics_parsed[9],      # checking for multiple dogs
    pics_parsed[28],     # checking for oil painting
    pics_parsed[37],     # checking for cartoon, BARANBY
    pics_parsed[49],     # checking for APRIL 2026
    pics_parsed[55],     # checking for curled-up posure
    pics_parsed[59],     # checking for statue
    pics_parsed[62],     # checking for chalkboard text
    pics_parsed[77],     # checking for cartoon, speech bubble text
    pics_parsed[88]      # checking for watercolor
    )

## 4. Evaluation Queries

We will evaluate your index by running queries against it — queries similar in spirit to the examples below, but not identical.

**Important:** Define a function called `query_index(query, filter=None, k=5)` that takes a natural language query, an optional metadata filter string (e.g., `"breed = 'pug'"`), and returns the top-k results as a list of dicts with at least a `"filename"` key. We will call this function to test your system.

Filter strings use SQL `WHERE` clause syntax (which LanceDB supports natively), e.g., `"breed = 'pug'"`, `"style = 'cartoon'"`, `"has_text = true"`.

### Example semantic queries
```python
query_index("a happy dog at the beach", k=5)
query_index("a dog sitting in a cafe", k=5)
query_index("a pencil drawing of a dog", k=5)
```

### Example filtered queries
```python
query_index("dog", filter="breed = 'corgi'", k=20)
query_index("dog", filter="style = 'cartoon'", k=20)
query_index("dog", filter="setting = 'living_room'", k=20)
query_index("dog", filter="activity = 'eating'", k=20)
```

### Example combined queries
```python
query_index("dog in the snow", filter="breed = 'husky'", k=5)
query_index("cute dog posing", filter="style = 'pencil_sketch'", k=5)
query_index("running on the beach", filter="breed = 'dalmatian'", k=5)
```

Run these example queries and display results clearly — filenames, metadata, and scores.

## 5. Submission

- [ ] Create a new branch called `homework-6`
- [ ] Commit and push your work (notebook + CSV files in `temp/`)
- [ ] Create a Pull Request
- [ ] Merge the PR to main
- [ ] Submit the `.ipynb` file on Blackboard

**Note:** Commit the notebook and the `temp/` files that your code created. The `temp/` directory is gitignored by default, so you will need to force-add it: `git add -f temp/`.